<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/circuit_tracer_examples/concept_direction_latent_dynamics.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# Concept-Direction Latent Dynamics

Where along the forward pass does a concept direction live? This notebook builds the color-vs-fruit concept direction
of the [concept-direction steering notebook](ct_concept_steering_demo.ipynb) (the `orange` sense-disambiguation
example) from each state `gemma-3-1b-it` computes on the way to its answer, and compares those directions stage by
stage:

- **Embedding stages** build the direction from token embeddings: the two target tokens (`Fruit`, `Color`), the mean
  embeddings of the concept words, and their paired rejection.
- **Answer-state stages** build it from the residual stream at the answer position of a set of probe prompts, one
  group per sense.
- **Context-state stages** build it from the residual stream at the concept word in context, its projection onto the
  answer states, and their paired residuals.

Each stage's direction is then examined four ways: how the directions align with each other, what each one does when
added after the final norm (a readout of its alignment with the unembedding), what each does when added mid-model
(causal steering), and which transcoder features its attribution graph selects. A final section projects the
per-example states and the stage directions into two dimensions.

The analysis itself lives in the `speediedan/concept-direction` Hub component
(`exp/analysis/concept_direction_latent_dynamics.py`), fetched at a pinned revision. This notebook runs it once and
lays out its report.

**Requirements:** a bf16 CUDA GPU (about 4.5 GB peak), the gated `google/gemma-3-1b-it` weights, the `examples` extra
(`plotly`, `umap-learn`) and the `git-deps` group (circuit-tracer).

In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
# -- The experiment component ---------------------------------------------------------------------
# Pinned, not floating: the revision you read is the revision that executes.
COMPONENT_REPO = "speediedan/concept-direction"
COMPONENT_REVISION = "f454a0312acad29db7f7b3743f50c719f8d7a595"
EXPERIMENT_KEY = "gemma3_1b_it_local_latent_dynamics_color_fruit_orange_fs_l10_n5"
# False resolves the revision from the local components cache only: offline, after an earlier fetch.
FETCH_COMPONENT = True
# The analysis is Python that component publishes, run in this kernel. Left False, the fetch cell stops with the
# reason and where the fetched code sits, rather than opting in on your behalf.
TRUST_COMPONENT_CODE = False
# Rows shown in the top-features table for each stage.
TOP_FEATURES_SHOWN = 5

## 1. Fetch the experiment, then run it

`it.hub.pull_experiment` fetches one experiment definition and the files it owns, pinned to one commit, and returns
the snapshot directory those files sit in (`it.hub.load_experiment` resolves the same from the local cache, without
the network). Nothing executes until the trust decision below; the analysis then builds
its own session from the experiment configuration (model, transcoders, prompts, stages), so this cell is the whole
run. Expect a few minutes on a single GPU: every stage builds a direction, sweeps two steering sites and computes an
attribution graph.

In [4]:
# @title 1: Fetch and run { display-mode: "form" }
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

import interpretune as it

for _env_candidate in (Path.cwd() / ".env", Path.home() / "repos" / "interpretune" / ".env"):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

if FETCH_COMPONENT:
    canonical_key, _, snapshot_dir = it.hub.pull_experiment(COMPONENT_REPO, EXPERIMENT_KEY, revision=COMPONENT_REVISION)
else:
    canonical_key, _, snapshot_dir = it.hub.load_experiment(COMPONENT_REPO, EXPERIMENT_KEY, revision=COMPONENT_REVISION)
print(f"fetched {COMPONENT_REPO}:{canonical_key} into {snapshot_dir.name}")

if not TRUST_COMPONENT_CODE:
    raise SystemExit(
        "Set TRUST_COMPONENT_CODE = True to continue.\n"
        f"The experiment is already cached at {snapshot_dir}, so you can read "
        "exp/analysis/concept_direction_latent_dynamics.py before deciding."
    )
os.environ["IT_TRUST_REMOTE_CODE"] = "1"
sys.path.insert(0, str(snapshot_dir))

from exp.analysis.concept_direction_latent_dynamics import run_latent_dynamics_analysis  # noqa: E402

# The fetch checks that the configuration's filename matches its key, so the key names exactly one file.
config_paths = sorted(snapshot_dir.rglob(f"{canonical_key}.yaml"))
assert len(config_paths) == 1, f"expected one {canonical_key}.yaml in the snapshot, found {config_paths}"
report = run_latent_dynamics_analysis(config_paths[0])
print(f"{len(report['stages'])} stages, {len(report['examples'])} probe examples")
print(f"store-state reconstruction check, largest difference: {report['reconstruction_max_abs_diff']:.3g}")

src/interpretune/hub/trust.py:96: Refusing to execute analysis ops from 'speediedan/jlens_steering_ops': loading it runs code published by that repo inside this process, with your filesystem, network and Hugging Face token. Interpretune does not execute hub-resident code unless you opt in.
  Inspect it first: interpretune.hub.pull('speediedan/jlens_steering_ops') caches the repo without executing anything, so you can read the entrypoint the manifest names.
  Then opt in for this session: os.environ['IT_TRUST_REMOTE_CODE'] = '1' (or export IT_TRUST_REMOTE_CODE=1 before starting the process).
  Pin a revision when you pull so trusted code cannot change under you.
  Details: https://interpretune.org/en/latest/usage/hub_trust_posture.html
src/interpretune/analysis/ops/dispatcher.py:1110: Alias 'ct_feature_intervention' already has an assigned op or alias so the alias specified by 'feature_intervention_forward' cannot be mapped to it
src/interpretune/analysis/ops/dispatcher.py:1110: Alias '

concept_direction.py:   0%|          | 0.00/152k [00:00<?, ?B/s]

fetched speediedan/concept-direction:gemma3_1b_it_local_latent_dynamics_color_fruit_orange_fs_l10_n5 into f454a0312acad29db7f7b3743f50c719f8d7a595


~/.cache/huggingface/interpretune/hub/components/models--speediedan--concept-direction/snapshots/f454a0312acad29db7f7b3743f50c719f8d7a595/exp/concept_direction.py:887: Falling back to public Neuronpedia because the local Neuronpedia webapp or DB is unavailable.


src/interpretune/config/shared.py:281: Could not find an auto-composition for <class 'interpretune.config.module.ITConfig'> that supports all of the following kwargs: {'nnsight_cfg': NNsightConfig(model_name='google/gemma-3-1b-it', device_map='cuda', torch_dtype='float32', dispatch=True, tokenizer_kwargs={'padding_side': 'left', 'add_bos_token': True}, trust_remote_code=False, attn_implementation=None, default_padding_side='left', remote=False, api_key=None), 'circuit_tracer_cfg': CircuitTracerConfig(backend='nnsight', model_name='google/gemma-3-1b-it', transcoder_set='mwhanna/gemma-scope-2-1b-it/transcoder_all/width_16k_l0_small_affine', dtype=torch.bfloat16, max_n_logits=10, desired_logit_prob=0.95, batch_size=64, max_feature_nodes=None, offload=None, freeze_attention=True, freeze_norms=True, lazy_encoder=None, lazy_decoder=True, lazy_delta_rank=None, tune_base=False, verbose=True, default_node_threshold=0.8, default_edge_threshold=0.98, save_graphs=True, graph_output_dir=None, analy

[INFO] interpretune.utils.logging: Loading ReplacementModel with backend: nnsight


... (412 outputs left out)

Attribution completed in 4.52s


~/.venvs/it_971/lib/python3.13/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
~/.venvs/it_971/lib/python3.13/site-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


9 stages, 6 probe examples
store-state reconstruction check, largest difference: 0


~/.venvs/it_971/lib/python3.13/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
~/.venvs/it_971/lib/python3.13/site-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## 2. The stages and the probe examples

Every stage is one concept direction: its **basis** (token embeddings, or residual-stream store states), its **mode**
(a plain mean difference between the two groups, or a paired rejection that removes what each pair shares), and, for
the store stages, the **raw norm** of the group separation before it is normalized.

The example table is the sanity check behind everything else: each probe prompt's expected answer, the model's margin
for it (the expected answer's logit minus the other target's), and the weight the example carries. The store keeps
only the examples the model answers as expected (`store_concept_correct_only: true` in this configuration) and weights
each by its margin; an example answered the other way would otherwise pull the example-weighted stages toward the
wrong sense, with a weight as large as its error. The `unweighted` stage counts every kept example equally.

In [5]:
# @title 2: Stage and example tables { display-mode: "form" }
import pandas as pd
from IPython.display import display

stage_table = pd.DataFrame(
    [
        {"stage": name, "basis": rec["basis"], "mode": rec["mode"], "raw norm": rec.get("raw_norm")}
        for name, rec in report["stages"].items()
    ]
).set_index("stage")
display(stage_table.style.format({"raw norm": "{:.1f}"}, na_rep=""))  # embed stages have no store separation

example_columns = [
    "example_index",
    "group",
    "expected_answer",
    "expected_answer_margin",
    "answered_as_expected",
    "weight",
    "answer_context_cosine",
]
examples = pd.DataFrame(report["examples"])[example_columns].set_index("example_index")
display(
    examples.style.format({"expected_answer_margin": "{:+.2f}", "weight": "{:.2f}", "answer_context_cosine": "{:.3f}"})
)
wrong = examples[~examples["answered_as_expected"]]
print(f"{len(examples) - len(wrong)} of {len(examples)} probe examples answered as expected")
for index, row in wrong.iterrows():
    print(
        f"  example {index} ({row['group']}): margin {row['expected_answer_margin']:+.2f}, weight {row['weight']:.2f}"
    )

,basis,mode,raw norm
stage,,,
target_token_embed_difference,embed,mean_difference,
embed_mean_difference,embed,mean_difference,
answer_state_mean_difference,store,mean_difference,59.7
context_state_mean_difference,store,mean_difference,74.8
projected_context_mean_difference,store,mean_difference,30.1
paired_residual_unweighted,store,paired_rejection,27.8
store_context_paired_rejection,store,paired_rejection,28.5
embed_paired_rejection,embed,paired_rejection,
store_answer_paired_rejection,store,paired_rejection,58.9


,group,expected_answer,expected_answer_margin,answered_as_expected,weight,answer_context_cosine
example_index,,,,,,
0,Fruit,Fruit,+18.88,True,18.88,0.315
1,Fruit,Fruit,+16.88,True,16.88,0.283
2,Fruit,Fruit,+15.38,True,15.38,0.224
3,Fruit,Fruit,+16.00,True,16.00,0.302
4,Color,Color,+14.12,True,14.12,0.292
5,Color,Color,+0.50,True,0.50,0.279


6 of 6 probe examples answered as expected


## 3. How the stage directions align

Cosine similarity between every pair of stage directions, all in the residual stream's coordinates. Blocks along the
diagonal are stages that agree on the concept; near-zero entries between blocks mean two stages read the same concept
along directions that have nothing in common.

In [6]:
# @title 3: Stage cosine matrix { display-mode: "form" }
import plotly.express as px

cosine_stages = report["cosine"]["stages"]
fig = px.imshow(
    report["cosine"]["matrix"].float().numpy(),
    x=cosine_stages,
    y=cosine_stages,
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    text_auto=".2f",
    title="Cosine similarity between stage directions",
)
fig.update_layout(height=650, width=850)
fig.show()

## 4. Steering: a readout after the final norm, and a causal edit at block 10

The target gap is $\mathrm{logit}(\text{Fruit}) - \mathrm{logit}(\text{Color})$; the prompt asks about `orange` and
the model answers `Color`, so it starts negative.

**At `unembed.hook_in`** each unit direction is added at fixed scales after the final norm. Nothing downstream can
transform it there, so the gap change is exactly linear in scale: this column measures how well each direction aligns
with the difference of the two target unembeddings, not what it does to the computation.

**At block 10** each stage is recomputed from the states at that hook and added there, at fractions of the stage's own
group separation at that site (residual norms there are hundreds, so fixed absolute scales would move little). This is
the causal column: the edit passes through every later layer.

In [7]:
# @title 4: Steering tables { display-mode: "form" }
readout = pd.DataFrame(
    [
        {"stage": name, "scale": rec["scale_factor"], "gap change": rec["gap_delta"]}
        for name, recs in report["steering"].items()
        for rec in recs
    ]
).pivot(index="stage", columns="scale", values="gap change")
print(f"readout at {report['steering_site']}: gap change per scale")
display(readout.style.format("{:+.2f}").background_gradient(cmap="RdBu_r", axis=None, vmin=-40, vmax=40))

causal = report["causal_steering"]
causal_table = pd.DataFrame(
    [
        {"stage": name, "fraction of separation": rec["fraction"], "gap change": rec["gap_delta"]}
        for name, recs in causal["efficacy"].items()
        for rec in recs
    ]
).pivot(index="stage", columns="fraction of separation", values="gap change")
print(f"\ncausal steering at {causal['hook']}: gap change per fraction of each stage's separation")
display(causal_table.style.format("{:+.2f}").background_gradient(cmap="RdBu_r", axis=None, vmin=-20, vmax=20))

readout at unembed.hook_in: gap change per scale


scale,10.000000,25.000000,50.000000,100.000000
stage,,,,
answer_state_mean_difference,-0.50,-1.00,-2.25,-4.25
context_state_mean_difference,+0.50,+1.00,+2.00,+4.00
embed_mean_difference,+3.00,+7.25,+15.00,+29.75
embed_paired_rejection,+2.50,+6.75,+13.50,+26.75
paired_residual_unweighted,+0.25,+0.50,+0.75,+1.75
projected_context_mean_difference,+0.50,+1.00,+2.00,+4.00
store_answer_paired_rejection,-0.50,-1.25,-2.75,-5.00
store_context_paired_rejection,+0.50,+0.75,+1.50,+3.00
target_token_embed_difference,+12.50,+31.50,+63.00,+126.38



causal steering at blocks.10.hook_out: gap change per fraction of each stage's separation


fraction of separation,0.250000,0.500000,1.000000,2.000000
stage,,,,
answer_state_mean_difference,+1.25,+3.00,+6.25,-0.28
context_state_mean_difference,+0.50,+2.50,+7.50,+16.69
embed_mean_difference,+0.75,+2.25,+5.62,+2.88
embed_paired_rejection,+1.50,+3.50,+6.25,+10.00
paired_residual_unweighted,-0.75,+0.50,+5.50,+14.62
projected_context_mean_difference,+0.50,+2.25,+5.75,+19.38
store_answer_paired_rejection,+1.25,+2.25,+3.50,+1.92
store_context_paired_rejection,-0.50,+0.75,+5.25,+14.88
target_token_embed_difference,+1.75,+3.50,+9.25,+34.56


## 5. Which features each stage's attribution graph selects

For each stage, one attribution graph with the stage's direction as its target, its top features (layers 10 and up,
ranked by absolute signed influence), and the gap change when those features are intervened on. Features are
`(layer, position, index)`; the position indexes the chat-rendered prompt, decoded beside it.

In [8]:
# @title 5: Per-stage top features { display-mode: "form" }
from transformers import AutoTokenizer

graphs = report.get("stage_graphs") or {}
assert graphs, "the experiment ran without stage_graphs: enable LATENT_DYNAMICS.stage_graphs to fill this section"
prompt_ids = next(iter(graphs.values()))["rendered_prompt_token_ids"]
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

feature_rows = []
for name, graph in graphs.items():
    for rank, ((layer, pos, index), score) in enumerate(zip(graph["feature_ids"], graph["feature_scores"])):
        if rank >= TOP_FEATURES_SHOWN:
            break
        feature_rows.append(
            {
                "stage": name,
                "gap change": f"{graph['gap_delta']:+.2f}",
                "rank": rank + 1,
                "layer": layer,
                "position": f"{pos} {tokenizer.decode([prompt_ids[pos]])!r}",
                "feature": index,
                "influence": score,
            }
        )
features = pd.DataFrame(feature_rows).set_index(["stage", "gap change", "rank"])
display(features.style.format({"influence": "{:+.3g}"}))

## 6. Projections

Two UMAP projections. The first places every probe example's states (answer position, context, projected and final
store state) in one plane, coloured by state and marked by sense group, so the path each example takes from context to
answer is visible. The second places the stage directions themselves.

In [9]:
# @title 6: Trajectory and direction projections { display-mode: "form" }
trajectory = report["projections"]["trajectory"]
fig = px.scatter(
    pd.DataFrame(trajectory["records"]),
    x="x",
    y="y",
    color="state",
    symbol="group",
    hover_data=["example_index"],
    title=f"Per-example latent states ({trajectory['method']}, {trajectory['backend']})",
)
fig.update_layout(height=550, width=850)
fig.show()

directions = report["projections"]["directions"]
fig = px.scatter(
    pd.DataFrame(directions["records"]),
    x="x",
    y="y",
    color="basis",
    symbol="mode",
    text="stage",
    title=f"Stage directions ({directions['method']}, {directions['backend']})",
)
fig.update_traces(textposition="top center")
fig.update_layout(height=550, width=850)
fig.show()

## What to look for

- **Three families, not one direction.** The embedding, answer-state and context-state stages form separate blocks in
  the cosine matrix: the same concept is carried along unrelated directions at different points in the computation.
- **Readout is not causation.** The final-site column ranks the embedding stages far above the store stages, because
  they are built from (or close to) the target unembeddings. At block 10 the store stages produce effects of the same
  order as the embedding stages once each is scaled by its own separation.
- **Shared features, opposite effects.** The answer-state and context-state graphs share their mid-layer features at
  the answer position and differ in their late features, which is where the sign of their effect is decided.

The measured numbers for this configuration are discussed on
[issue #971](https://github.com/speediedan/interpretune/issues/971).